In [1]:
import torch
inputs = torch.tensor([
    [0.43,0.15,0.89], #Your->x^1
    [0.55,0.87,0.66], #journey->x^2
    [0.57,0.85,0.64], #starts->x^3
    [0.22,0.58,0.33], #with->x^4
    [0.77,0.25,0.10], #one->x^5
    [0.05,0.80,0.55] #step-> x^6
])

In [44]:
import torch.nn as nn

### Computing Context Vector for token at Position 2

In [7]:
x_2 = inputs[1]
d_in =  inputs.shape[1] #Input Embedding Size
d_out = 2 #Output Embedding Size

In [8]:
#Setting random seed for reproducibility of results

torch.manual_seed(123)

In [9]:
#We use nn.Parameter to initialize Weight matrices W_q,W_k and W_v.
#We set requires_grad= False, since right now we dont want the weight matrices to be updated 

In [10]:
W_q = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad= False)
W_k = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad= False)
W_v = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad= False)

In [11]:
#Shape of W_q, W_k, W-V Matrices
print(f"Shape of W_q; {W_q.shape}")
print(f"Shape of W_k; {W_k.shape}")
print(f"Shape of W_v; {W_v.shape}")

Shape of W_q; torch.Size([3, 2])
Shape of W_k; torch.Size([3, 2])
Shape of W_v; torch.Size([3, 2])


In [12]:
W_q

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])

In [14]:
# We want to calculate context vector corresponding to "journey"
# So we calculate query vector corresponding to inputs[1] embedding

In [16]:
query_2 = inputs[1] @ W_q 

In [17]:
query_2

tensor([0.4306, 1.4551])

#### Interesting Note: View this as linear transformation of inputs[1] embeding from 3 -D space to 2-D Space

In [18]:
#We also need to obtain key and value matrices

In [19]:
#Since we need to find similarity of query_2 with all tokens..
#..We calculate key and value matrices across all the input embeddings

In [20]:
keys = inputs@ W_k
values = inputs @ W_v

In [22]:
print(f"Shape of keys Matrix: {keys.shape}")
print(f"Shape of values Matrix: {values.shape}")


Shape of keys Matrix: torch.Size([6, 2])
Shape of values Matrix: torch.Size([6, 2])


In [23]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])

In [25]:
#Attention score of query_2 with key corresponding to token at index 3

as_2_3= query_2.dot(keys[3])
print(as_2_3)

tensor(1.0795)


In [26]:
#Attention Scores for query_2 with all the keys
#Which is basically a dot product between query_2 and each of the vectors in key matric

attention_score_2 = query_2 @ keys.T 
print(attention_score_2)


tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


#### Calculating Attention Weights

In [31]:
d_k = keys.shape[-1]
attn_weights_2= torch.softmax(attention_score_2/d_k**0.5,dim=-1) #compute softmax corr. to last dimension 

In [32]:
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])

#### Calculating Context Vector

In [36]:
context_vec_2 = attn_weights_2 @ values

In [37]:
print(context_vec_2)

tensor([0.3061, 0.8210])


## Implementing a compact self-attention class for future use:

In [47]:
class SelfAttention_v1(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias=False): 
        
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out, bias = qkv_bias)

    def forward(self,x):
        
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        context_vec = attn_weights @ values
        
        return context_vec
    

    

In [48]:
torch.manual_seed(789)
sa_v1 = SelfAttention_v1(d_in=3,d_out=2) #intializing sa_v1 object
print(sa_v1(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


#### ....Thus we get 6 context vectors, one corresponding to each token 